In [ ]:
# Library install
# %pip install semantic-link-labs

# package load now handled by Env

In [ ]:
# library setup
import notebookutils 
import sempy.fabric as fabric
import sempy_labs as labs
import sempy_labs.admin as admin
import sempy_labs.graph as graph
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from azure.identity import ClientSecretCredential

In [ ]:
# get the env variables
env_vars = notebookutils.variableLibrary.getLibrary("fabric-management-vars")

In [ ]:
# environment params
common_capacity = env_vars.getVariable('common_capacity')
fabric_admins = env_vars.getVariable('fabric_admins')

# set up to mount containers
storage_account_name = env_vars.getVariable('admin_storage_account')
container_name = env_vars.getVariable('admin_filepath')
mount_path = "/" + storage_account_name
lakehouse_path = "abfss://" + container_name + "@" + storage_account_name + ".dfs.core.windows.net"
data_types = ['Files', 'Tables']

# service principal authentication setup
key_vault_uri = env_vars.getVariable('admin_keyvault_uri') # Enter your key vault URI
tenant_id = env_vars.getVariable('tenant_id')
## The Admin SP
admin_sp_client_id = env_vars.getVariable('admin_sp_app_id')
admin_sp_client_secret_name = (key_vault_uri, "PBI-FabricAdminEditAPI-ITS-FabricManagement")
## The calling SP
orchestration_sp_client_id =env_vars.getVariable('orchestration_sp_app_id')
orchestration_sp_app_object_id = env_vars.getVariable('orchestration_sp_app_object_id')

# dynamic vars
active_workspace_name = fabric.resolve_workspace_name()
active_workspace_id = fabric.get_workspace_id()
## get ID because pipeline fails on getName - bc running as a service principal
executing_user = mssparkutils.env.getUserId()

In [ ]:
# Mount external datalake "admin_lake" into specified local path
mssparkutils.fs.mount(
    lakehouse_path,
    mount_path
)
# check there's stuff in there
mssparkutils.fs.ls(f"file://{mssparkutils.fs.getMountPath(mount_path)}")

In [ ]:
# adding conditional logic as not all necessary Admin APIs currently support Service Principal authentication - in future we can remove this check
## If authing as a user, skips this check
if orchestration_sp_app_object_id == executing_user: 
    # Use SP context - requires Keyvault access to read secrets(PIM or via SP caller)
    # go get the keyvault value
    try:
        client_secret = notebookutils.credentials.getSecret(
            key_vault_uri, 
            admin_sp_client_secret_name[1]
        )
        credential = labs.ServicePrincipalTokenProvider(client_secret)
        credential_name = labs.ServicePrincipalTokenProvider(admin_sp_client_secret_name[1])
        credential_csc = ClientSecretCredential(tenant_id, admin_sp_client_id, admin_sp_client_secret_name[1])
        print("Keyvault authentication successful!")
    except: 
        notebook_config_auth_failure_dict = {
                "message": "Authentication failed: check executing entity's access to referenced keyvault or activate PIM"
            }   
        mssparkutils.notebook.exit(notebook_config_auth_failure_dict)
# close IF ELSE

In [ ]:
# adding conditional logic as not all necessary Admin APIs currently support Service Principal authentication - in future we can remove this check 
if orchestration_sp_app_object_id == executing_user: 
    # Use SP context - requires Keyvault access to read secrets(PIM or via SP caller)
    # go get the keyvault value
    try:
        with labs.service_principal_authentication(
            key_vault_uri = key_vault_uri,
            key_vault_tenant_id = 'tenant-fabric-sp',
            key_vault_client_id = 'fabric-admin-sp-app-id', 
            key_vault_client_secret = admin_sp_client_secret_name[1]
            ):
            df = admin.list_capacities()
            print("Service Principal Authorisation successful")
    except: 
        notebook_config_auth_failure_dict = {
                "message": "Authorisation failed: PBI API call failed as Service Prinicpal, refer to Error code"
            }   
        mssparkutils.notebook.exit(notebook_config_auth_failure_dict)
else:
    # use User context (requires Fabric Admin)
    try:
        df = admin.list_capacities()
        print("User Authorisation successful")
    except: 
        notebook_config_auth_failure_dict = {
                "message": "Authorisation failed: PBI API call failed as User, refer to Error code"
            }   
        mssparkutils.notebook.exit(notebook_config_auth_failure_dict)
# close IF ELSE